In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter, periodogram
from scipy.special import erf
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
CSV_FILE           = r'C:\Users\Asus\OneDrive - Indian Institute of Technology Guwahati\Desktop\Mini Project\final data an images\1hz low data.csv'
SAVE_DIR           = r'C:\Users\Asus\OneDrive - Indian Institute of Technology Guwahati\Desktop\Mini Project\final data an images'   # folder for output PNGs
SEQ_START          = 1
SEQ_END            = 1200
WAVELENGTH_NM      = 516
DOWNSAMPLE         = 1
ZC_SMOOTH_WINDOW   = 11
MIN_GOOD_FRINGES   = 20
ERF_AGREE_THR      = 0.30
DPI                = 200
# ═══════════════════════════════════════════════════════════════════════════════

import os
os.makedirs(SAVE_DIR, exist_ok=True)

def savefig(fig, name):
    path = os.path.join(SAVE_DIR, name)
    fig.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"    Saved: {path}")

print("=" * 65)
print("BEAM WAIST MEASUREMENT — ZERO-CROSSING CALIBRATION PIPELINE")
print("=" * 65)

# ── LOAD ──────────────────────────────────────────────────────────────────────
print("\n[1/6] Loading raw CSV...")
with open(CSV_FILE, 'r') as f:
    lines = f.readlines()
parts       = lines[1].strip().split(',')
t_start_s   = float(parts[3])
t_increment = float(parts[4])
raw     = np.genfromtxt(CSV_FILE, delimiter=',', skip_header=2, usecols=(0, 1, 2))
seq     = raw[:, 0].astype(int)
knife   = raw[:, 1]
fringe  = raw[:, 2]
time_s  = t_start_s + seq * t_increment
time_ms = time_s * 1000.0

mask = (seq >= SEQ_START) & (seq <= SEQ_END)
t    = time_ms[mask][::DOWNSAMPLE]
k    = knife[mask][::DOWNSAMPLE]
fr   = fringe[mask][::DOWNSAMPLE]
N    = len(t)
print(f"    {N} samples  ({t[0]:.2f} to {t[-1]:.2f} ms)")

# ── ZERO-CROSSING CALIBRATION ─────────────────────────────────────────────────
print("\n[2/6] Zero-crossing fringe calibration...")
fr_ac  = fr - np.mean(fr)
fr_sm  = savgol_filter(fr_ac, window_length=ZC_SMOOTH_WINDOW, polyorder=3)
signs  = np.sign(fr_sm)
signs[signs == 0] = 1
zc_idx = np.where(np.diff(signs))[0]

half_fringes  = len(zc_idx)
total_fringes = half_fringes / 2.0
total_disp_nm = total_fringes * (WAVELENGTH_NM / 2.0)

print(f"    Zero crossings : {half_fringes}  ({total_fringes:.1f} fringes)")
print(f"    Total scan     : {total_disp_nm/1000:.3f} µm")
if total_fringes < MIN_GOOD_FRINGES:
    print(f"    WARNING: only {total_fringes:.0f} fringes — position unreliable.")

# Build position axis
if half_fringes >= 2:
    zc_pos_nm  = np.arange(half_fringes + 1) * (WAVELENGTH_NM / 4.0)
    anchor_idx = np.concatenate([[0], zc_idx, [N - 1]])
    anchor_pos = np.concatenate([[0], zc_pos_nm[1:half_fringes + 1], [total_disp_nm]])
    pos_nm     = np.interp(np.arange(N), anchor_idx, anchor_pos)
else:
    pos_nm = np.linspace(0, total_disp_nm, N)
    print("    Fallback: constant-speed position assumed.")

# ── NORMALISE KNIFE-EDGE ──────────────────────────────────────────────────────
print("\n[3/6] Normalizing knife-edge...")
k_sorted = np.sort(k)
P_min    = np.mean(k_sorted[:max(1, int(0.05 * N))])
P_max    = np.mean(k_sorted[int(0.95 * N):])
P_norm   = np.clip((k - P_min) / (P_max - P_min), 0, 1)
if P_norm[0] > P_norm[-1]:
    P_norm = 1.0 - P_norm

baseline_val    = np.mean(P_norm[:max(1, N // 20)])
plateau_val     = np.mean(P_norm[-max(1, N // 20):])
has_flat_bottom = baseline_val < 0.05
has_flat_top    = plateau_val  > 0.95
print(f"    Baseline : {baseline_val:.4f}  {'OK' if has_flat_bottom else 'MISSING'}")
print(f"    Plateau  : {plateau_val:.4f}  {'OK' if has_flat_top else 'MISSING'}")

# ── WIDTH MEASUREMENTS ────────────────────────────────────────────────────────
print("\n[4/6] Width measurements...")
sort_idx = np.argsort(pos_nm)
pos_s    = pos_nm[sort_idx]
pn_s     = P_norm[sort_idx]

x_10   = np.interp(0.10,   pn_s, pos_s)
x_50   = np.interp(0.50,   pn_s, pos_s)
x_90   = np.interp(0.90,   pn_s, pos_s)
x_1585 = np.interp(0.1585, pn_s, pos_s)
x_8415 = np.interp(0.8415, pn_s, pos_s)

# CORRECTED CONSTANTS FOR BEAM RADIUS (w0)
w0_1090 = abs(x_90 - x_10) / 1.2815
w0_1e2  = abs(x_8415 - x_1585) # No division! The distance IS w0

widths   = np.array([w0_1090, w0_1e2])
w_mean   = np.mean(widths)
w_spread = np.max(widths) - np.min(widths)

print(f"    w₀ (10-90%) : {w0_1090/1000:.3f} µm")
print(f"    w₀ (1/e²)   : {w0_1e2/1000:.3f} µm")
print(f"    Spread      : {w_spread/1000:.3f} µm  "
      f"({'GOOD' if w_spread < 0.3*w_mean else 'WARNING: large spread'})")

# ── ERF FIT ───────────────────────────────────────────────────────────────────
print("\n[5/6] Error function fit...")
erf_fit_ok = erf_fit_valid = False

def erf_model(x, A, x0, w0, B):
    return (A / 2.0) * (1 + erf((x - x0) / (abs(w0) / np.sqrt(2)))) + B

try:
    popt, pcov = curve_fit(
        erf_model, pos_nm, P_norm,
        p0=[1.0, x_50, w0_1090, 0.0],
        bounds=([0.7, pos_nm.min(), w0_1090*0.3, -0.10],
                [1.3, pos_nm.max(), w0_1090*2.0,  0.10]),
        maxfev=200000, method='trf'
    )
    w0_erf     = abs(popt[2])
    w0_erf_err = np.sqrt(np.diag(pcov))[2]
    x0_erf     = popt[1]
    erf_fit_ok = True
    ratio      = w0_erf / w0_1090
    erf_fit_valid = (abs(ratio - 1.0) < ERF_AGREE_THR) and (total_fringes >= MIN_GOOD_FRINGES)
    print(f"    w₀ = {w0_erf/1000:.3f} ± {w0_erf_err/1000:.3f} µm  "
          f"[{'VALID' if erf_fit_valid else 'UNRELIABLE'}]  ratio={ratio:.3f}")
except Exception as e:
    print(f"    Erf fit failed: {e}")

# ── RELIABILITY ───────────────────────────────────────────────────────────────
if erf_fit_ok and erf_fit_valid:
    primary_w0, primary_err = w0_erf/1000.0, w0_erf_err/1000.0
    primary_method, reliability = "Erf fit", "HIGH"
elif total_fringes >= MIN_GOOD_FRINGES:
    primary_w0, primary_err = w0_1090/1000.0, w_spread/2000.0
    primary_method, reliability = "10-90% width", "MODERATE"
else:
    primary_w0, primary_err = w0_1090/1000.0, w0_1090*0.30/1000.0
    primary_method, reliability = "10-90% width", "LOW — increase scan amplitude"

print(f"\n    PRIMARY: w₀ = {primary_w0:.3f} ± {primary_err:.3f} µm  [{reliability}]")

color_map   = {'HIGH': 'darkgreen', 'MODERATE': 'darkorange',
               'LOW — increase scan amplitude': 'darkred'}
title_color = color_map.get(reliability, 'darkred')

# ════════════════════════════════════════════════════════════════
print("\n[6/6] Saving individual plots...")

# ── PLOT 1: Raw signals ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax2 = ax.twinx()
ax.plot(t, fr_sm, color='mediumorchid', lw=0.8, label='Fringe (AC, smoothed)')
ax2.plot(t, k, color='darkcyan', lw=0.8, alpha=0.8, label='Knife-edge')
ax.set_xlabel('Time (ms)', fontsize=11)
ax.set_ylabel('Fringe AC (V)', color='mediumorchid', fontsize=11)
ax2.set_ylabel('Knife-edge (V)', color='darkcyan', fontsize=11)
ax.set_title(f'Raw Signals — Fringe & Knife-Edge\n'
             f'({total_fringes:.0f} fringes counted, scan = {total_disp_nm/1000:.2f} µm)',
             fontsize=12)
ax.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, '01_raw_signals.png')

# ── PLOT 2: Position axis ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, pos_nm / 1000, color='steelblue', lw=1.2)
ax.set_xlabel('Time (ms)', fontsize=11)
ax.set_ylabel('Position (µm)', fontsize=11)
ax.set_title('Position Axis from Zero-Crossing Interferometric Calibration\n'
             f'({total_fringes:.0f} fringes, λ = {WAVELENGTH_NM} nm)', fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, '02_position_axis.png')

# ── PLOT 3: Fringe power spectrum ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fs_hz   = 1.0 / (t_increment * DOWNSAMPLE)
f_hz, psd = periodogram(fr_sm, fs=fs_hz)
ax.semilogy(f_hz, psd, color='steelblue', lw=0.8)
if len(f_hz) > 1:
    peak_f = f_hz[np.argmax(psd[1:]) + 1]
    ax.axvline(peak_f, color='red', lw=1.5, linestyle='--',
               label=f'Peak: {peak_f:.1f} Hz  →  {peak_f*(WAVELENGTH_NM/2)/1000:.1f} nm/s')
    ax.legend(fontsize=9)
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('PSD (V²/Hz)', fontsize=11)
ax.set_title('Fringe Power Spectrum (mirror speed diagnostic)', fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, '03_fringe_spectrum.png')

# ── PLOT 4: S-curve ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(pos_nm / 1000, P_norm, color='steelblue', s=6, alpha=0.5,
           label='Measured data', zorder=3)
x_plt = np.linspace(pos_nm.min(), pos_nm.max(), 3000)
if erf_fit_ok:
    y_fit = erf_model(x_plt, *popt)
    lbl   = (f'Erf fit:  w₀ = {w0_erf/1000:.3f} µm  '
             f'[{"VALID" if erf_fit_valid else "UNRELIABLE"}]')
    ax.plot(x_plt / 1000, y_fit, 'k-', lw=2.0, label=lbl)
    ax.axvline(x0_erf / 1000, color='darkorange', lw=1.5, linestyle='--',
               label=f'Beam centre = {x0_erf/1000:.3f} µm')
ax.axvline(x_10 / 1000, color='royalblue', lw=1.2, linestyle=':',
           label=f'10% level = {x_10/1000:.3f} µm')
ax.axvline(x_90 / 1000, color='royalblue', lw=1.2, linestyle='-.',
           label=f'90% level = {x_90/1000:.3f} µm')
ax.axvspan(x_10 / 1000, x_90 / 1000, alpha=0.07, color='royalblue')
ax.axhline(0.1585, color='gray', lw=0.8, linestyle=':', alpha=0.6,
           label='1/e² levels (15.85%, 84.15%)')
ax.axhline(0.8415, color='gray', lw=0.8, linestyle=':', alpha=0.6)
ax.set_xlabel('Position (µm)', fontsize=12)
ax.set_ylabel('Normalized Power', fontsize=12)
ax.set_title(f'Knife-Edge S-Curve — λ = {WAVELENGTH_NM} nm\n'
             f'Primary result: w₀ = {primary_w0:.3f} ± {primary_err:.3f} µm  [{reliability}]',
             fontsize=12, color=title_color)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, '04_scurve.png')

# ── PLOT 5: Method comparison bar chart ───────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
method_names = ['10-90%\nwidth', '1/e²\ndirect']
w0_vals      = [w0_1090/1000, w0_1e2/1000]
bar_cols     = ['steelblue', 'steelblue']
if erf_fit_ok and erf_fit_valid:
    method_names.append('Erf\nfit')
    w0_vals.append(w0_erf/1000)
    bar_cols.append('darkgreen')
bars = ax.bar(method_names, w0_vals, color=bar_cols, edgecolor='black', lw=0.8)
ax.axhline(primary_w0, color='red', lw=2, linestyle='--',
           label=f'Primary: {primary_w0:.3f} µm')
for bar, val in zip(bars, w0_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002*max(w0_vals),
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('w₀ (µm)', fontsize=12)
ax.set_title('Beam Waist: Comparison of Methods', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
savefig(fig, '05_method_comparison.png')

# ── PLOT 6: Results summary (text panel) ──────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
rows = [
    ("PRIMARY RESULT", "", ""),
    (f"  Method      : {primary_method}", "", ""),
    (f"  w\u2080          : {primary_w0:.3f} \u00b1 {primary_err:.3f} \u00b5m", "", "PRIMARY"),
    (f"  Reliability : {reliability}", "", ""),
    ("", "", ""),
    ("ALL METHODS", "w\u2080 (\u00b5m)", ""),
    ("\u2500"*24, "\u2500"*10, ""),
    ("10-90% width",    f"{w0_1090/1000:.3f}", ""),
    ("1/e\u00b2 direct",   f"{w0_1e2/1000:.3f}", ""),
]
if erf_fit_ok:
    rows.append(("Erf fit",
                 f"{w0_erf/1000:.3f} \u00b1 {w0_erf_err/1000:.3f}",
                 "VALID" if erf_fit_valid else "UNRELIABLE"))
rows += [
    ("\u2500"*24, "\u2500"*10, ""),
    ("Scan range",    f"{total_disp_nm/1000:.2f} \u00b5m", ""),
    ("Fringes (ZC)",  f"{total_fringes:.0f}", ""),
    ("Wavelength",    f"{WAVELENGTH_NM} nm", ""),
    ("Flat bottom",   "YES" if has_flat_bottom else "NO", ""),
    ("Flat top",      "YES" if has_flat_top    else "NO", ""),
]
y = 0.97
col_prim = color_map.get(reliability, 'darkred')
for row in rows:
    c = col_prim if row[2] == "PRIMARY" else \
        'darkred'   if row[2] == "UNRELIABLE" else \
        'darkgreen' if row[2] == "VALID" else 'black'
    fw = 'bold' if row[2] in ("PRIMARY", "UNRELIABLE", "VALID") else 'normal'
    ax.text(0.03, y, row[0], transform=ax.transAxes,
            fontsize=10, fontfamily='monospace', va='top', color=c, fontweight=fw)
    ax.text(0.70, y, row[1], transform=ax.transAxes,
            fontsize=10, fontfamily='monospace', va='top', color=c, fontweight=fw)
    y -= 0.075
ax.set_title('Measurement Summary', fontsize=12, pad=10)
fig.tight_layout()
savefig(fig, '06_summary.png')

# ── PLOT 7: Gaussian Beam Profile (Derivative) ────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# 1. Smooth the S-curve data slightly more to prepare for derivative
pn_smooth = savgol_filter(pn_s, window_length=51, polyorder=3)

# 2. Calculate gradient
dy_dx = np.gradient(pn_smooth)

# 3. Suppress boundary artifacts (ignore the first and last 20 points)
dy_dx[:20] = 0
dy_dx[-20:] = 0
dy_dx[dy_dx < 0] = 0 

# 4. Normalize based on the TRUE peak, avoiding edge artifacts
dy_dx_norm = dy_dx / np.max(dy_dx) 

# 5. Smooth the final derivative for a clean visual
dy_dx_norm = savgol_filter(dy_dx_norm, window_length=31, polyorder=3)
dy_dx_norm[dy_dx_norm < 0] = 0 # Re-clip tails to 0 after smoothing

# Plot the numerical derivative (Faint line WITH discrete data points)
ax.plot(pos_s / 1000, dy_dx_norm, color='steelblue', lw=1.0, alpha=0.4, zorder=1)
ax.plot(pos_s / 1000, dy_dx_norm, color='steelblue', marker='.', linestyle='None', 
        markersize=4, alpha=0.8, zorder=2, label='Numerical Derivative (Data points)')

# Calculate Theoretical Fit boundaries
if erf_fit_ok:
    x_theoretical = x_plt / 1000
    gauss_fit = np.exp(-2 * ((x_plt - x0_erf) / w0_erf)**2)
    ax.plot(x_theoretical, gauss_fit, 'k--', lw=2.0, zorder=3, label='Theoretical Gaussian Fit')
    center_um = x0_erf / 1000
    w0_um = w0_erf / 1000
else:
    center_um = x_50 / 1000
    w0_um = w0_1090 / 1000

# Mark 1/e^2 horizontal level
e2_level = np.exp(-2) # Approx 0.1353
ax.axhline(e2_level, color='gray', linestyle=':', lw=1.2, zorder=0,
           label='1/e² intensity level (13.5%)')

# Mark boundaries with vertical dashed lines
left_bound = center_um - w0_um
right_bound = center_um + w0_um
ax.axvline(left_bound, color='darkorange', linestyle='--', lw=1.2, zorder=0, label=f'-w₀ boundary')
ax.axvline(right_bound, color='darkorange', linestyle='--', lw=1.2, zorder=0, label=f'+w₀ boundary')

ax.set_xlabel('Position (µm)', fontsize=12)
ax.set_ylabel('Normalized Intensity', fontsize=12)
ax.set_title(f'Gaussian Beam Profile\n(Derived from S-Curve, w₀ ≈ {w0_um:.2f} µm)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, '07_gaussian_profile.png')

# ── FINAL REPORT ──────────────────────────────────────────────────────────────
print()
print("╔══════════════════════════════════════════════════════════════╗")
print("║            FINAL BEAM WAIST REPORT (ZC PIPELINE)             ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  PRIMARY w₀  =  {primary_w0:8.3f} µm  ±  {primary_err:.3f} µm               ║")
print(f"║  Method      :  {primary_method:<44} ║")
print(f"║  Reliability :  {reliability:<44} ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  10-90% → w₀ =  {w0_1090/1000:6.3f} µm                               ║")
print(f"║  1/e²   → w₀ =  {w0_1e2/1000:6.3f} µm                               ║")
if erf_fit_ok:
    print(f"║  Erf fit → w₀ = {w0_erf/1000:6.3f} ± {w0_erf_err/1000:.3f} µm  "
          f"[{'VALID' if erf_fit_valid else 'UNRELIABLE'}]         ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Fringes (ZC)  : {total_fringes:6.0f}                                ║")
print(f"║  Total scan    : {total_disp_nm/1000:6.2f} µm                               ║")
print(f"║  Flat bottom   : {'YES' if has_flat_bottom else 'NO':<36} ║")
print(f"║  Flat top      : {'YES' if has_flat_top    else 'NO':<36} ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"\nAll plots saved to: {SAVE_DIR}")

BEAM WAIST MEASUREMENT — ZERO-CROSSING CALIBRATION PIPELINE

[1/6] Loading raw CSV...
    1199 samples  (-791.50 to -192.50 ms)

[2/6] Zero-crossing fringe calibration...
    Zero crossings : 353  (176.5 fringes)
    Total scan     : 45.537 µm

[3/6] Normalizing knife-edge...
    Baseline : 0.0341  OK
    Plateau  : 0.9571  OK

[4/6] Width measurements...
    w₀ (10-90%) : 14.993 µm
    w₀ (1/e²)   : 15.986 µm
    Spread      : 0.993 µm  (GOOD)

[5/6] Error function fit...
    w₀ = 13.169 ± 0.106 µm  [VALID]  ratio=0.878

    PRIMARY: w₀ = 13.169 ± 0.106 µm  [HIGH]

[6/6] Saving individual plots...
    Saved: C:\Users\Asus\OneDrive - Indian Institute of Technology Guwahati\Desktop\Mini Project\final data an images\01_raw_signals.png
    Saved: C:\Users\Asus\OneDrive - Indian Institute of Technology Guwahati\Desktop\Mini Project\final data an images\02_position_axis.png
    Saved: C:\Users\Asus\OneDrive - Indian Institute of Technology Guwahati\Desktop\Mini Project\final data an images\